# K-Means 2D Clustering Hardware Accelerator
Running on PYNQ board using the kmeans_axi_wrapper IP

**Register Map:**
| Offset | Description |
|--------|-------------|
| 0x00 | CTRL - bit[0]=start (self-clearing) |
| 0x04 | STATUS - bit[0]=done, [11:8]=iter_count |
| 0x08 | PX_LO - points 0-3 x-coords |
| 0x0C | PX_HI - points 4-7 x-coords |
| 0x10 | PY_LO - points 0-3 y-coords |
| 0x14 | PY_HI - points 4-7 y-coords |
| 0x18 | RESULT - {cy1,cx1,cy0,cx0} |
| 0x1C | DIST0 debug |

In [ ]:
from pynq import Overlay, MMIO
import time

# Load the bitstream onto the FPGA
ol = Overlay('kmeans_accel.bit')
print('Bitstream loaded successfully!')

In [ ]:
# Set up MMIO access to the kmeans IP
KMEANS_BASE  = 0x40000000
KMEANS_RANGE = 0x10000

kmeans = MMIO(KMEANS_BASE, KMEANS_RANGE)

# Register offsets
REG_CTRL   = 0x00
REG_STATUS = 0x04
REG_PX_LO  = 0x08
REG_PX_HI  = 0x0C
REG_PY_LO  = 0x10
REG_PY_HI  = 0x14
REG_RESULT = 0x18
REG_DIST0  = 0x1C

print('MMIO configured at 0x{:08X}'.format(KMEANS_BASE))

In [ ]:
def run_kmeans(px, py):
    """
    Run hardware K-Means clustering on 8 points.
    px: list of 8 x-coordinates (0-255)
    py: list of 8 y-coordinates (0-255)
    Returns: (cx0, cy0, cx1, cy1, iterations)
    """
    assert len(px) == 8 and len(py) == 8, "Need exactly 8 points"
    
    # Pack 4 x 8-bit coords into each 32-bit word
    px_lo = (px[0] | (px[1] << 8) | (px[2] << 16) | (px[3] << 24)) & 0xFFFFFFFF
    px_hi = (px[4] | (px[5] << 8) | (px[6] << 16) | (px[7] << 24)) & 0xFFFFFFFF
    py_lo = (py[0] | (py[1] << 8) | (py[2] << 16) | (py[3] << 24)) & 0xFFFFFFFF
    py_hi = (py[4] | (py[5] << 8) | (py[6] << 16) | (py[7] << 24)) & 0xFFFFFFFF
    
    # Write point data
    kmeans.write(REG_PX_LO, px_lo)
    kmeans.write(REG_PX_HI, px_hi)
    kmeans.write(REG_PY_LO, py_lo)
    kmeans.write(REG_PY_HI, py_hi)
    
    # FIX: If FSM is in ST_DONE from a previous run,
    # one start pulse moves it to ST_IDLE but then it needs
    # ANOTHER pulse to proceed to ST_INIT.
    # Solution: always send two start pulses with a gap.
    if kmeans.read(REG_STATUS) & 0x1:   # done=1 means FSM is in ST_DONE
        kmeans.write(REG_CTRL, 0x1)     # pulse 1: ST_DONE -> ST_IDLE
        time.sleep(0.001)               # wait for FSM to settle
    
    # Pulse START: ST_IDLE -> ST_INIT -> runs
    kmeans.write(REG_CTRL, 0x1)
    
    # Poll until done (with timeout ~5 seconds)
    timeout = 50000
    while not (kmeans.read(REG_STATUS) & 0x1):
        timeout -= 1
        if timeout == 0:
            return None, None, None, None, -1
        time.sleep(0.0001)
    
    # Read results
    result     = kmeans.read(REG_RESULT)
    status     = kmeans.read(REG_STATUS)
    
    cx0 = (result >>  0) & 0xFF
    cy0 = (result >>  8) & 0xFF
    cx1 = (result >> 16) & 0xFF
    cy1 = (result >> 24) & 0xFF
    iterations = (status >> 8) & 0xF
    
    return cx0, cy0, cx1, cy1, iterations

print('run_kmeans() ready (with re-run fix)')

In [ ]:
# -------------------------------------------------------
# TEST 1: Two clear clusters
# Cluster A: points around (20, 20)
# Cluster B: points around (200, 200)
# -------------------------------------------------------
px = [15, 20, 25, 18,   195, 200, 205, 198]
py = [15, 20, 25, 18,   195, 200, 205, 198]

cx0, cy0, cx1, cy1, iters = run_kmeans(px, py)

if iters == -1:
    print('ERROR: Timeout - check base address and bitstream')
else:
    print(f'=== K-Means Result ===')
    print(f'Centroid 0: ({cx0}, {cy0})')
    print(f'Centroid 1: ({cx1}, {cy1})')
    print(f'Iterations: {iters}')

In [ ]:
# -------------------------------------------------------
# TEST 2: Visualize with matplotlib (re-run works now!)
# -------------------------------------------------------
import matplotlib.pyplot as plt

px = [15, 20, 25, 18,   195, 200, 205, 198]
py = [15, 20, 25, 18,   195, 200, 205, 198]

cx0, cy0, cx1, cy1, iters = run_kmeans(px, py)

if iters == -1:
    print('ERROR: Timeout!')
else:
    fig, ax = plt.subplots(figsize=(6,6))
    ax.scatter(px[:4], py[:4], c='blue', s=100, label='Cluster 0 points')
    ax.scatter(px[4:], py[4:], c='red',  s=100, label='Cluster 1 points')
    ax.scatter([cx0], [cy0], c='blue', s=300, marker='*', label=f'Centroid 0 ({cx0},{cy0})')
    ax.scatter([cx1], [cy1], c='red',  s=300, marker='*', label=f'Centroid 1 ({cx1},{cy1})')
    ax.set_xlim(0, 255)
    ax.set_ylim(0, 255)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(f'K-Means Hardware Result (iters={iters})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# -------------------------------------------------------
# TEST 3: Different clusters - diagonal
# -------------------------------------------------------
px = [10, 12, 14, 11,   240, 238, 242, 245]
py = [240, 238, 242, 245,   10, 12, 14, 11]

cx0, cy0, cx1, cy1, iters = run_kmeans(px, py)

print(f'=== Diagonal Clusters ===')
print(f'Centroid 0: ({cx0}, {cy0})  [expected ~(12, 241)]')
print(f'Centroid 1: ({cx1}, {cy1})  [expected ~(241, 12)]')
print(f'Iterations: {iters}')

In [ ]:
import time

# ── Software K-Means (pure Python) ──────────────────────
def software_kmeans(px, py, max_iter=15):
    cx0, cy0 = px[0], py[0]   # init from first points
    cx1, cy1 = px[1], py[1]
    
    for iteration in range(max_iter):
        # Assign clusters
        clusters = []
        for i in range(len(px)):
            d0 = (px[i]-cx0)**2 + (py[i]-cy0)**2
            d1 = (px[i]-cx1)**2 + (py[i]-cy1)**2
            clusters.append(0 if d0 <= d1 else 1)
        
        # Update centroids
        s0x,s0y,n0 = 0,0,0
        s1x,s1y,n1 = 0,0,0
        for i in range(len(px)):
            if clusters[i] == 0: s0x+=px[i]; s0y+=py[i]; n0+=1
            else:                 s1x+=px[i]; s1y+=py[i]; n1+=1
        
        new_cx0 = s0x//n0 if n0 else cx0
        new_cy0 = s0y//n0 if n0 else cy0
        new_cx1 = s1x//n1 if n1 else cx1
        new_cy1 = s1y//n1 if n1 else cy1
        
        if (new_cx0,new_cy0,new_cx1,new_cy1) == (cx0,cy0,cx1,cy1):
            break
        cx0,cy0,cx1,cy1 = new_cx0,new_cy0,new_cx1,new_cy1
    
    return cx0,cy0,cx1,cy1

# ── Test Data ────────────────────────────────────────────
px = [15, 20, 25, 18, 195, 200, 205, 198]
py = [15, 20, 25, 18, 195, 200, 205, 198]
RUNS = 1000  # average over 1000 runs

# ── Benchmark Software ───────────────────────────────────
t0 = time.perf_counter()
for _ in range(RUNS):
    software_kmeans(px, py)
t_sw = (time.perf_counter() - t0) / RUNS * 1e6  # microseconds

# ── Benchmark Hardware ───────────────────────────────────
t0 = time.perf_counter()
for _ in range(RUNS):
    run_kmeans(px, py)
t_hw = (time.perf_counter() - t0) / RUNS * 1e6  # microseconds

# ── Results ──────────────────────────────────────────────
print("=" * 45)
print(f"  Software (Python):  {t_sw:8.2f} µs per run")
print(f"  Hardware (FPGA):    {t_hw:8.2f} µs per run")
print(f"  Speedup:            {t_sw/t_hw:8.1f}x faster")
print("=" * 45)
print(f"\n  Note: Hardware compute is ~260 ns")
print(f"  Remaining time is AXI register overhead")
